# Factor: close_execution_chunkiness_shift — AI-GENERATED (Claude research workflow; NOT the Gemini miner pipeline)

**Provenance (disclosed, distinct from our other AI factors):** this factor's
logic was designed by an Anthropic Claude agent (factor-designer role) inside
the team's multi-agent research workflow on 2026-07-27, then adversarially
reviewed by a second Claude agent, and evaluated unchanged on-platform. It did
NOT come from the team's Gemini evolutionary miner (`kit/agent_miner_book.py`);
its audit trail is `design_provenance.md` in this folder (design JSON +
adversarial critique, verbatim) and AAD §5/§6. Human-written parts: the data
plumbing boilerplate (month-chunked `main()`, gates), reused verbatim from the
verified 07 template.

**Hypothesis:** average deal size (volume / deal_number) is a
retail-vs-institutional trade-granularity proxy untouched by any of our live
factors. The factor contrasts late-session (14:27–14:56, closing auction
excluded) average deal size against the stock's own earlier-day baseline,
bounded in (−1, 1). Raw orientation read NEGATIVE on-platform → at the
submitted SIGN = −1, late-session FRAGMENTATION (retail chase into the close)
predicts positive next-day cross-sectional returns — consistent with the
A-share smart-money/reversal literature on per-trade-size factors.

**Platform eval (bigalpha_eval, 2024, terminal run 2026-07-27,
`endgame/chunkiness.log`):** raw +1 read ic_mean −0.009233 / ic_ir −0.270935 /
LS sharpe −1.542595 / stress_ic_ir −0.433334 → SIGN flipped once to −1 per the
mandatory first-run check. At submit orientation: **ic +0.0092, icir +0.271,
stress_ic_ir +0.433, LS sharpe +1.543** — all four Part-A legs positive; the
strongest stress leg the team has measured.

**Measured orthogonality (H1-2024 mean daily Spearman, submitted signs,
`endgame/corr_chunk.log`):** corr to 05 vov = +0.012, to 06 orderly = −0.012,
to 07 order_size_imbalance = −0.004 — orthogonal to the entire live book.
Coverage: min 90.0% / mean 97.9% (late-window sparsity measured, clears the
60% gate; deal_number 97.6% positive per-minute).

**Contract:** identical to the verified 07 template — `main(datasources,
start_date, end_date)`, month-chunked `dai.query` on `datasources["bar1m"]`
({table} placeholder, no hardcoded bar table), gates, output
`date/instrument/factor` with no inf; eval via `M.bigalpha_eval._latest` on
2024. Coverage note: per AAD §6.2 the in-notebook coverage gate is computed
post-fill in this template generation; coverage was verified out-of-band
(min 90%).

In [ ]:
SIGN = -1.0        # VERIFIED on-platform 2026-07-27: raw +1 eval read ic_mean=-0.0092 -> flipped once
                       # (eval ic_mean) and flip once if negative -- never
                       # submit unverified.

# ================== BEGIN AI-GENERATED FACTOR LOGIC ==================
# Factor logic designed by the Claude research-workflow agent 2026-07-27;
# name: close_execution_chunkiness_shift
# cluster: M
# rationale: Measures institutional versus retail order flow imbalance by comparing average order sizes on bid and ask sides with dynamic fallback guards.
# sign_hypothesis: +1
FACTOR_SQL = """
WITH r AS (
  SELECT date::DATE AS dd, instrument,
         EXTRACT(HOUR FROM date)*100 + EXTRACT(MINUTE FROM date) AS hm,
         CAST(volume AS DOUBLE) AS v,
         CAST(deal_number AS DOUBLE) AS dn
  FROM {table}
  WHERE ask_price1 > 0 AND bid_price1 > 0
    AND volume > 0 AND deal_number > 0
),
agg AS (
  SELECT dd, instrument,
         SUM(CASE WHEN hm >= 1427 AND hm <= 1456 THEN v ELSE 0.0 END) AS v_late,
         SUM(CASE WHEN hm >= 1427 AND hm <= 1456 THEN dn ELSE 0.0 END) AS n_late,
         SUM(CASE WHEN hm < 1427 THEN v ELSE 0.0 END) AS v_base,
         SUM(CASE WHEN hm < 1427 THEN dn ELSE 0.0 END) AS n_base
  FROM r
  GROUP BY dd, instrument
),
ads AS (
  SELECT dd, instrument,
         v_late / NULLIF(n_late, 0.0) AS ads_late,
         v_base / NULLIF(n_base, 0.0) AS ads_base
  FROM agg
)
SELECT dd::DATETIME AS date, instrument,
       (ads_late - ads_base) / NULLIF(ads_late + ads_base, 0.0) AS factor
FROM ads
ORDER BY date, instrument
"""
# =================== END AI-GENERATED FACTOR LOGIC ===================


def main(datasources, start_date, end_date):
    """Evaluator substitutes datasources/start_date/end_date and calls this.
    datasources (dict): {logical name: physical table}; we use "bar1m".
    Returns pd.DataFrame['date','instrument','factor'], no inf."""
    import pandas as pd
    import numpy as np
    import dai

    bar1m = datasources["bar1m"]
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    # month-chunked querying: the factor is WITHIN-DAY, so chunking is exact
    parts = []
    for m in pd.period_range(start, end, freq="M"):
        cs = max(m.start_time, start).strftime("%Y-%m-%d 00:00:00")
        ce = min(m.end_time, end).strftime("%Y-%m-%d 23:59:59")
        part = dai.query(FACTOR_SQL.format(table=bar1m),
                         filters={"date": [cs, ce]}, compression=True).df()
        parts.append(part)
    f = pd.concat(parts, ignore_index=True)

    f["date"] = pd.to_datetime(f["date"])
    f["instrument"] = f["instrument"].astype(str)
    f["factor"] = SIGN * pd.to_numeric(f["factor"], errors="coerce")
    f["factor"] = f["factor"].where(np.isfinite(f["factor"]))   # no inf

    cons = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                     filters={"date": [start_date, end_date]}).df()
    cons["date"] = pd.to_datetime(cons["date"])
    cons["instrument"] = cons["instrument"].astype(str)
    out = cons.merge(f, on=["date", "instrument"], how="left")
    out = out.loc[(out["date"] >= start) & (out["date"] <= end)].copy()
    # degenerate books (frozen/limit-locked): no computable value = neutral
    out["factor"] = out["factor"].fillna(0.0)

    exp_days = cons.loc[(cons["date"] >= start) & (cons["date"] <= end),
                        "date"].drop_duplicates()
    missing = exp_days[~exp_days.isin(out["date"].unique())]
    if len(missing) > 0:
        raise ValueError(f"GATE FAIL: {len(missing)} trading days missing")
    cov = out.groupby("date")["factor"].apply(lambda s: s.notna().mean()).min()
    if not (cov >= 0.60):
        raise ValueError(f"GATE FAIL: worst per-day coverage {cov:.1%} < 60%")
    return out[["date", "instrument", "factor"]]

In [ ]:
# ===== evaluation (exact official factor_hf pattern) =====
if __name__ == '__main__':
    from bigmodule import M
    import dai

    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    factor_data = main(datasources, start_date, end_date)
    print(factor_data.shape, factor_data['date'].min(), factor_data['date'].max())

    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )